# **OpenCV'de bir Caffe Modeli kullanarak Siyah Beyaz Fotoğrafları Renklendirme**

**Bu derste Siyah Beyaz (gri tonlamalı) bir Fotoğrafı otomatik olarak Renklendirmek için önceden eğitilmiş modelleri nasıl kullanacağımızı öğreneceğiz**


### **Derin Öğrenme ile Siyah Beyaz Fotoğraflara Hayat Vermek**
Siyah beyaz fotoğrafların büyülü dünyasına renk katmak, uzun zamandır fotoğrafçılık ve yapay zeka alanında ilgi uyandıran bir konuydu. Richard Zhang, Phillip Isola ve Alexei A. Efros tarafından 2016 yılında yayımlanan "Colorful Image Colorization" adlı makale, bu zorlu göreve derin öğrenme ile şaşırtıcı bir çözüm getirdi.

<img src="teaser3.jpg" width="600">

Bu çalışma, siyah beyaz bir görüntüyü renklendirme sürecindeki belirsizliği (yani bir ton siyah-beyazdan birden fazla olası renge ulaşma ihtimalini) bir sınıflandırma problemi olarak ele alıyor. Yazarlar, bu yaklaşımı benimseyerek, ortaya çıkan renk çeşitliliğini artırmak için eğitim aşamasında özel bir sınıf dengeleme tekniği kullanıyorlar.

Nasıl Çalışıyor?

Bu yöntem, Evrişimsel Sinir Ağları (CNN) adı verilen derin öğrenme modellerini kullanıyor. Bir milyondan fazla renkli görüntü üzerinde eğitilen bu ağ, test aşamasında siyah beyaz bir fotoğrafı alıyor ve tek bir ileri besleme geçişiyle o fotoğrafa renkli bir görünüm kazandırıyor.

Aşağıdaki Caffe model dosyalarını kullanmak gerekir:

    - colorization_deploy_v2.prototext: Modelin mimarisini ve katmanlarını tanımlar.

    - colorization_release_v2.caffe: Eğitilmiş modelin ağırlıklarını içerir.

    - pts_in_hull.npy: Renk uzayında kullanılan, renk sınıflandırmasına yardımcı olan verileri içerir.



In [ ]:
import cv2
import numpy as np
from os import listdir
from os.path import isfile, join
from matplotlib import pyplot as plt

# Define our imshow function 
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

**Caffe, Berkeley Vision and Learning Center (BVLC) tarafından geliştirilen, derin öğrenme için tasarlanmış bir açık kaynaklı yapıdır.** 
Özellikle bilgisayarlı görü (computer vision) alanındaki uygulamalar için popülerdir.

Caffe'nin temel özellikleri şunlardır:\
- Hız: Caffe, özellikle GPU hızlandırmasıyla yüksek hızda çalışmak üzere tasarlanmıştır. Bu sayede, büyük veri kümeleri üzerinde model eğitme ve test etme süreçleri daha kısa sürede tamamlanabilir.\
- İfade Gücü: Model mimarilerini (katmanları, bağlantıları vb.) kolayca tanımlamaya ve değiştirmeye olanak tanıyan basit ve esnek bir yapı sunar. Bu tanımlar genellikle metin tabanlı .prototxt dosyalarıyla yapılır.\
- Modülerlik: Her bir katman, ayrı bir modül olarak ele alınır. Bu sayede, önceden tanımlanmış katmanları bir araya getirerek karmaşık ağ yapıları oluşturmak veya kendi özel katmanlarınızı eklemek oldukça kolaydır.\
- Topluluk: Geniş ve aktif bir topluluğa sahiptir. Bu, birçok önceden eğitilmiş modelin ve ilgili kaynakların kolayca bulunabileceği anlamına gelir.


OpenCV (cv2) kütüphanesi, Caffe tarafından eğitilmiş modelleri doğrudan destekler.

OpenCV'nin derin öğrenme modülü olan DNN (Deep Neural Network) modülü, Caffe'nin yanı sıra TensorFlow, PyTorch, ONNX ve Darknet gibi popüler derin öğrenme çerçevelerinden eğitilmiş modelleri yükleme ve kullanma yeteneğine sahiptir.

In [ ]:
# Script https://github.com/richzhang/colorization/blob/master/colorize.py'a dayanmaktadır.
# Caffemodel ve prototxt'i indirmek için bkz: https://github.com/richzhang/colorization/tree/master/models
# pts_in_hull.npy dosyasını indirmek için bkz: https://github.com/richzhang/colorization/blob/master/resources/pts_in_hull.npy

# Resimlerimizi alalım, colorize klasörü içinde farklı resimler bulunmaktadır
file_path = "../files/colorize/blackandwhite/" 
blackandwhite_imgs = [f for f in listdir(file_path) if isfile(join(file_path, f))] # Klasördeki tüm resim dosyalarını listeye alıyor.
kernel = '../files/colorize/pts_in_hull.npy' #Renk kümelerini içeren pts_in_hull.npy dosyasının yolu.
# Bu dosya, modelin renklendirme için kullandığı renk kümesi merkezlerini içerir. Model, çıktısını bu renk sınıflarından birine atar.



# Select desired model
net = cv2.dnn.readNetFromCaffe("../files/colorize/colorization_deploy_v2.prototxt",
                           "../files/colorize/colorization_release_v2.caffemodel")
# OpenCV’nin dnn modülü ile Caffe modelini yüklüyor:
#  .prototxt: Modelin mimarisini tanımlar.
#  .caffemodel: Modelin ağırlıklarını içerir.

# NumPy kütüphanesi kullanılarak, daha önce bahsedilen pts_in_hull.npy dosyasındaki renk kümesi verileri belleğe yüklenir.
pts_in_hull = np.load(kernel) 

# np.load() fonksiyonu, NumPy’nin .npy formatındaki dosyalarını yükler.
# Yani bu dosyadaki veriler, Python/NumPy dizisi (array) olarak belleğe aktarılır.

pts_in_hull = pts_in_hull.transpose().reshape(2, 313, 1, 1)   # pts_in_hull.npy, 313 renk merkezinden oluşan bir küme içeriyor.
# Bu satırlar, yüklenen pts_in_hull verilerini sinir ağına, özellikle class8_ab adlı son katmana yerleştirir. 
# Bu, modelin tahmin ettiği renklerin hangi renklere karşılık geldiğini bilmesini sağlar.
net.getLayer(net.getLayerId('class8_ab')).blobs = [pts_in_hull.astype(np.float32)]
net.getLayer(net.getLayerId('conv8_313_rh')).blobs = [np.full([1, 313], 2.606, np.float32)]

for image in blackandwhite_imgs: # döngüsü ile her bir siyah-beyaz fotoğraf için sırayla çalışır.
    img = cv2.imread(file_path+image) #  İşlenecek fotoğrafı okur.
    
    img_rgb = (img[:,:,[2, 1, 0]] * 1.0 / 255).astype(np.float32)
    img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2LAB)
    # Fotoğrafın renk uzayını BGR'den (Mavi-Yeşil-Kırmızı) Lab’ye* dönüştürür. 
    # *Lab renk uzayı, insan gözünün renk algısına daha yakın bir modeldir.
    # L kanalı (Lightness): Parlaklık veya açıklık/koyuluk bilgisini içerir.
    # a kanalı: Yeşilden kırmızıya geçişi temsil eder.
    # b kanalı: Maviden sarıya geçişi temsil eder.

    
    # Fotoğrafın sadece L kanalını (parlaklık) alır.
    img_l = img_lab[:,:,0]  
    
    # Sinir ağı modelleri genellikle sabit bir giriş boyutu bekler (bu model için 224x224). 
    # Bu satır, orijinal fotoğrafın boyutunu küçültür.
    (H_orig,W_orig) = img_rgb.shape[:2] 

    # Sinir ağı modelleri genellikle sabit bir giriş boyutu bekler (bu model için 224x224). 
    # Bu satır, orijinal fotoğrafın boyutunu küçültür.
    img_rs = cv2.resize(img_rgb, (224, 224)) 

    
    img_lab_rs = cv2.cvtColor(img_rs, cv2.COLOR_RGB2Lab) # küçültülmüş görüntünün L*a*b* versiyonunu içerir. 
    img_l_rs = img_lab_rs[:,:,0] # bu diziden sadece ilk kanalı (indeks 0) seçer. 
    # L*a*b* renk uzayında, bu ilk kanal parlaklık (L) kanalıdır.
    # Bu işlemin yapılmasının temel nedeni, kullanılan sinir ağının sadece parlaklık (L)  
    # kanalını girdi olarak alacak şekilde eğitilmiş olmasıdır.
    
    # Sinir ağının daha iyi performans göstermesi için L kanalındaki parlaklık değerlerinden 50 çıkarılarak 
    # ortalamanın sıfıra yakın olması sağlanır (ortalama merkezleme).
    img_l_rs -= 50 

    net.setInput(cv2.dnn.blobFromImage(img_l_rs))
    
    # bu bizim sonucumuz
    ab_dec = net.forward('class8_ab')[0,:,:,:].transpose((1,2,0)) 
    #  Sinir ağının tahmin ettiği a ve b renk kanalları, orijinal fotoğrafın boyutuna geri döndürülür.

    #Bu bölüm orijinal fotoğrafın L kanalı (parlaklık) ile sinir ağının tahmin ettiği a ve b kanallarını 
    #birleştirerek tam bir Lab* fotoğrafı oluşturur.
    (H_out,W_out) = ab_dec.shape[:2]
    ab_dec_us = cv2.resize(ab_dec, (W_orig, H_orig))
    img_lab_out = np.concatenate((img_l[:,:,np.newaxis],ab_dec_us),axis=2) 
    
    # Yeni oluşturulan Lab* fotoğrafı tekrar BGR (standart renk) formatına dönüştürülür
    img_bgr_out = np.clip(cv2.cvtColor(img_lab_out, cv2.COLOR_Lab2BGR), 0, 1)

    # orijinal görüntüyü göster
    imshow('Original', img,5)
    # Renklendirilmiş orijinal boyutlarına yeniden boyutlandır ve görüntüle 
    img_bgr_out = cv2.resize(img_bgr_out, (W_orig, H_orig), interpolation = cv2.INTER_AREA)
    imshow('Colorized', img_bgr_out,5)